In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg://fabriceg@localhost:5433/benin_agri")

df = pd.read_sql("SELECT * FROM analysis_master", engine)
print(df.shape)
df.head()

(1186, 7)


,crop,year,production_t,area_ha,yield_kg_ha,fertilizer_t,pesticide_t
0,Tomatoes,2012,216052.33,37968.0,5690.4,26754.0,1692.0
1,Karite nuts (sheanuts),1992,8000.00,3500.0,2285.7,15325.0,1334.0
2,"Seed cotton, unginned",1967,12273.00,29713.0,413.1,2495.0,NaN
3,"Seed cotton, unginned",1973,44618.00,54743.0,815.0,4376.0,NaN
4,Sugar cane,1993,34063.00,943.0,36122.0,17238.0,1184.0


In [2]:
# Filter out rows with no production or no yield
df_clean = df[(df["production_t"] > 0) & (df["yield_kg_ha"].notna())].copy()

# Keep only years with pesticide data (>= 1990) to have complete cases
df_clean = df_clean[df_clean["year"] >= 1990]

print(df_clean.shape)
print(df_clean["crop"].nunique(), "crops")
print(df_clean["year"].min(), "to", df_clean["year"].max())

(664, 7)
19 crops
1990 to 2024


In [3]:
# Overall correlation: do inputs move with yield?
corr_matrix = df_clean[["yield_kg_ha", "fertilizer_t", "pesticide_t"]].corr()
print(corr_matrix)

              yield_kg_ha  fertilizer_t  pesticide_t
yield_kg_ha      1.000000      0.032873     0.045967
fertilizer_t     0.032873      1.000000     0.828702
pesticide_t      0.045967      0.828702     1.000000


In [4]:
# Correlation per crop: does yield move with inputs within each crop?
results = []
for crop in df_clean["crop"].unique():
    sub = df_clean[df_clean["crop"] == crop]
    if len(sub) >= 10:  # need enough years
        r_fert = sub["yield_kg_ha"].corr(sub["fertilizer_t"])
        r_pest = sub["yield_kg_ha"].corr(sub["pesticide_t"])
        results.append({"crop": crop, "r_fertilizer": round(r_fert, 3), "r_pesticide": round(r_pest, 3), "n_years": len(sub)})

corr_df = pd.DataFrame(results).sort_values("r_fertilizer", ascending=False)
corr_df

,crop,r_fertilizer,r_pesticide,n_years
10,Soya beans,0.672,0.646,35
17,"Chillies and peppers, green (Capsicum spp. and...",0.610,0.707,35
5,"Cashew nuts, in shell",0.584,0.748,35
12,Rice,0.502,0.664,35
16,Oil palm fruit,0.457,0.615,35
13,"Cow peas, dry",0.447,0.519,34
14,Pineapples,0.416,0.522,35
0,Tomatoes,0.406,0.518,35
15,Okra,0.395,0.559,35
11,"Groundnuts, excluding shelled",0.345,0.331,35


In [6]:
import statsmodels.api as sm

results = []
for crop in df_clean["crop"].unique():
    sub = df_clean[df_clean["crop"] == crop].dropna(subset=["yield_kg_ha", "fertilizer_t", "pesticide_t"])
    if len(sub) < 10:
        continue
    X = sub[["fertilizer_t", "pesticide_t"]]
    X = sm.add_constant(X)
    y = sub["yield_kg_ha"]
    model = sm.OLS(y, X).fit()
    results.append({
        "crop": crop,
        "coef_fert": round(model.params["fertilizer_t"], 4),
        "pval_fert": round(model.pvalues["fertilizer_t"], 4),
        "coef_pest": round(model.params["pesticide_t"], 4),
        "pval_pest": round(model.pvalues["pesticide_t"], 4),
        "r_squared": round(model.rsquared, 3),
        "n": len(sub)
    })

reg_df = pd.DataFrame(results).sort_values("r_squared", ascending=False)
reg_df

,crop,coef_fert,pval_fert,coef_pest,pval_pest,r_squared,n
5,"Cashew nuts, in shell",-0.0003,0.5819,0.0382,0.0003,0.564,35
1,Karite nuts (sheanuts),0.0025,0.1178,-0.1407,0.0001,0.513,35
17,"Chillies and peppers, green (Capsicum spp. and...",0.0016,0.7289,0.2741,0.0071,0.501,35
10,Soya beans,0.0023,0.0646,0.0307,0.2241,0.477,35
12,Rice,-0.0033,0.5097,0.3411,0.0019,0.449,35
16,Oil palm fruit,-0.0129,0.5013,1.1910,0.0045,0.387,35
15,Okra,-0.0016,0.4058,0.1099,0.0075,0.327,35
14,Pineapples,-0.0166,0.8389,3.5120,0.0427,0.274,35
13,"Cow peas, dry",0.0001,0.8440,0.0240,0.0923,0.271,34
0,Tomatoes,-0.0020,0.7872,0.3193,0.0397,0.270,35


In [7]:
results = []
for crop in df_clean["crop"].unique():
    sub = df_clean[df_clean["crop"] == crop].dropna(subset=["yield_kg_ha", "fertilizer_t", "pesticide_t"])
    if len(sub) < 10:
        continue
    X = sub[["fertilizer_t", "pesticide_t", "year"]]
    X = sm.add_constant(X)
    y = sub["yield_kg_ha"]
    model = sm.OLS(y, X).fit()
    results.append({
        "crop": crop,
        "coef_fert": round(model.params["fertilizer_t"], 4),
        "pval_fert": round(model.pvalues["fertilizer_t"], 4),
        "coef_pest": round(model.params["pesticide_t"], 4),
        "pval_pest": round(model.pvalues["pesticide_t"], 4),
        "r_squared": round(model.rsquared, 3),
        "n": len(sub)
    })

reg_df2 = pd.DataFrame(results).sort_values("r_squared", ascending=False)
reg_df2

,crop,coef_fert,pval_fert,coef_pest,pval_pest,r_squared,n
16,Oil palm fruit,-0.0025,0.6985,-0.2375,0.1483,0.932,35
5,"Cashew nuts, in shell",-0.0000,0.9370,0.0053,0.3087,0.914,35
12,Rice,-0.0007,0.7439,-0.0168,0.7402,0.910,35
1,Karite nuts (sheanuts),0.0017,0.0545,-0.0382,0.0839,0.853,35
13,"Cow peas, dry",0.0006,0.1099,-0.0249,0.0087,0.811,34
17,"Chillies and peppers, green (Capsicum spp. and...",0.0036,0.2717,0.0025,0.9749,0.770,35
14,Pineapples,0.0215,0.6593,-1.7301,0.1589,0.752,35
10,Soya beans,0.0027,0.0069,-0.0304,0.2055,0.688,35
3,"Cassava, fresh",-0.0141,0.1637,-0.3158,0.2062,0.682,35
18,Maize (corn),-0.0002,0.6991,-0.0501,0.0032,0.611,35


In [8]:
reg_df2.to_sql("stats_regression", engine, if_exists="replace", index=False)
print("Saved stats_regression")

Saved stats_regression


In [9]:
decomp = []
for crop in df_clean["crop"].unique():
    sub = df_clean[df_clean["crop"] == crop].sort_values("year")
    first = sub.iloc[0]
    last = sub.iloc[-1]

    prod_change = last["production_t"] - first["production_t"]
    area_change = last["area_ha"] - first["area_ha"]
    yield_change = last["yield_kg_ha"] - first["yield_kg_ha"]

    # Contribution decomposition
    # Production = Area × Yield / 1000  (kg/ha → t/ha conversion)
    # Approximate: growth from area = yield_first × area_change
    # growth from yield = area_first × yield_change
    area_contrib = (first["yield_kg_ha"] * area_change) / 1000
    yield_contrib = (first["area_ha"] * yield_change) / 1000

    total_contrib = abs(area_contrib) + abs(yield_contrib)
    if total_contrib > 0:
        area_pct = round(100 * area_contrib / total_contrib, 1)
        yield_pct = round(100 * yield_contrib / total_contrib, 1)
    else:
        area_pct = yield_pct = None

    decomp.append({
        "crop": crop,
        "first_year": first["year"],
        "last_year": last["year"],
        "prod_change_t": round(prod_change, 0),
        "area_pct": area_pct,
        "yield_pct": yield_pct
    })

decomp_df = pd.DataFrame(decomp).sort_values("area_pct", ascending=False)
decomp_df

,crop,first_year,last_year,prod_change_t,area_pct,yield_pct
10,Soya beans,1990,2024,652107.0,100.0,0.0
4,"Seed cotton, unginned",1990,2024,489570.0,99.9,-0.1
1,Karite nuts (sheanuts),1990,2024,79292.0,98.7,-1.3
6,"Onions and shallots, dry (excluding dehydrated)",1990,2024,52970.0,98.1,1.9
2,Sugar cane,1990,2024,53380.0,96.3,-3.7
5,"Cashew nuts, in shell",1990,2024,209624.0,92.5,7.5
15,Okra,1990,2024,59120.0,91.8,8.2
18,Maize (corn),1990,2024,1684830.0,89.6,10.4
12,Rice,1990,2024,489665.0,89.6,10.4
14,Pineapples,1990,2024,456588.0,85.7,14.3


In [10]:
decomp_df.to_sql("stats_decomposition", engine, if_exists="replace", index=False)
print("Saved stats_decomposition")

Saved stats_decomposition
